# Data Loading

Data link:

https://www.phoenixopendata.com/dataset/officer-show-of-force/resource/7e9d5fc7-ce02-4108-80af-369b1b54c4ff

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json
import os

In [11]:
batch_size = 1000

# URL and parameters for the request
url = 'https://www.phoenixopendata.com/api/3/action/datastore_search'
params = {
    'resource_id' : '7e9d5fc7-ce02-4108-80af-369b1b54c4ff',
    'limit' : batch_size,
    'offset' : 0
}

# List holding the data
data = []

# The size of the last batch (intitial value of 1 to ensure the loop runs)
last_batch_size = 1

# Check if the CSV file already exists (if so, we can just use that instead of batch loading)
if os.path.exists('data.csv'):
    print('CSV file found - loading data from file')
    df = pd.read_csv('data.csv')
else:
    print('CSV file NOT found - loading data from API')

    # While there are still more rows to load
    while last_batch_size > 0:
        # Make request
        response = requests.get(url, params=params)
        response.raise_for_status()

        # Get batch from response JSON and add it to the overall data list
        current_batch = response.json()['result']['records']
        data.extend(current_batch)

        # Move up the offset based on the batch size
        params['offset'] = params['offset'] + batch_size

        # Update last batch size
        last_batch_size = len(current_batch)
        print(f'Loaded batch with size {last_batch_size}')

    # Convert to pandas dataframe and save to csv
    df = pd.DataFrame(data)
    df.to_csv('data.csv',index=False)

print('Done')

CSV file found - loading data from file
Done


In [13]:
df.iloc[1]

_id                                            2
INC_IR_NO                      202500000193276.0
INC_IA_NO                             SOF25-0006
INC_DATE                     2025-02-18T00:00:00
INC_YEAR                                    2025
INC_TIME                                   13:50
INC_DAY_WEEK                             Tuesday
INC_BEAT                                721 Beat
HUNDRED_BLOCK            6XXX West Mcdowell Road
INC_CITY                                 Phoenix
INC_STATE                                     AZ
INC_ZIPCODE                                85035
INC_PRECINCT          Maryvale/Estrella Precinct
CIT_NUMBER                               61390.0
CIT_GENDER                                  Male
CIT_AGE                                       29
SUBJ_AGE_GROUP                               20s
CIT_RACE                                   White
CIT_ETHNICITY                  Hispanic / Latino
SIMPLE_SUBJ_RE_GRP                         White
CITIZEN_CHARGE      

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4730 entries, 0 to 4729
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 4730 non-null   int64  
 1   INC_IR_NO           4727 non-null   float64
 2   INC_IA_NO           4730 non-null   object 
 3   INC_DATE            4730 non-null   object 
 4   INC_YEAR            4730 non-null   int64  
 5   INC_TIME            4726 non-null   object 
 6   INC_DAY_WEEK        4730 non-null   object 
 7   INC_BEAT            4730 non-null   object 
 8   HUNDRED_BLOCK       4730 non-null   object 
 9   INC_CITY            4730 non-null   object 
 10  INC_STATE           4730 non-null   object 
 11  INC_ZIPCODE         4730 non-null   object 
 12  INC_PRECINCT        4730 non-null   object 
 13  CIT_NUMBER          4728 non-null   float64
 14  CIT_GENDER          4730 non-null   object 
 15  CIT_AGE             4730 non-null   object 
 16  SUBJ_A

# Data Cleaning

Checking for duplicates and NA values:

In [ ]:
print(f'Number of duplicates: {df.duplicated().sum()}')

Number of duplicates: 0


In [ ]:
print(f'Number of rows with NA values: {df.isna().any(axis=1).sum()}')

Number of rows with NA values: 9


## Data Cleaning Tasks

| Column  | Task |
|---|---|
| _id  | N/A |
| INC_IR_NO | Convert to numeric |
| INC_IA_NO | N/A |
| INC_DATE | Convert to date format |
| INC_YEAR | N/A |
| INC_TIME | Convert to time format |
| INC_DAY_WEEK | N/A |
| INC_BEAT | Format values consistently |
| HUNDRED_BLOCK | N/A |
| INC_CITY | N/A |
| INC_STATE | N/A |
| INC_ZIPCODE | N/A |
| INC_PRECINCT | N/A |
| CIT_NUMBER | N/A |
| CIT_GENDER | N/A |
| CIT_AGE | Clean up unreasonable values |
| SUBJ_AGE_GROUP | Clean up unreasonable values |
| CIT_RACE | Clean up duplicate or ambiguous values |
| CIT_ETHNICITY | Clean up duplicate or ambiguous values |
| SIMPLE_SUBJ_RE_GRP | N/A |
| CITIZEN_CHARGE | N/A |
| HIGHEST_SHOW_FORCE | N/A |
| SHOW_FORCE_COUNT | N/A |

## Cleaning ```INC_IR_NO```

In [ ]:
df['INC_IR_NO'].unique()

array(['202500000255424', '202500000193276', '202500000252010', ...,
       '202501868103', '202501867554', '202501861673'], dtype=object)

We just need to convert the row to numeric format:

In [ ]:
df['INC_IR_NO'] = pd.to_numeric(df['INC_IR_NO'])

In [ ]:
df['INC_IR_NO'].unique()

array([2.02500000e+14, 2.02500000e+14, 2.02500000e+14, ...,
       2.02501868e+11, 2.02501868e+11, 2.02501862e+11])

## Cleaning ```INC_DATE```

In [ ]:
df['INC_DATE'].unique()[:10]

array(['2025-02-18T00:00:00', '2025-02-19T00:00:00',
       '2025-02-20T00:00:00', '2025-02-21T00:00:00',
       '2025-02-22T00:00:00', '2025-02-23T00:00:00',
       '2025-02-24T00:00:00', '2025-02-25T00:00:00',
       '2025-02-26T00:00:00', '2025-02-27T00:00:00'], dtype=object)

We just need to convert the dates from strings to actual date objects. Since pandas ```datetime``` objects store the time as well, we'll fill in the time using the values in the ```INC_TIME``` column.

In [ ]:
def fix_INC_DATE(row):
  INC_DATE = row['INC_DATE']
  INC_TIME = row['INC_TIME']

  if pd.isna(INC_DATE):
    return pd.NA

  # If INC_TIME exists
  if not pd.isna(INC_TIME):
    # Remove "T00:00:00" substring
    INC_DATE = INC_DATE.replace('T00:00:00', '')

    # Add the time from the INC_TIME column
    INC_DATE = INC_DATE + 'T' + INC_TIME + ':00'

  # Convert to date
  INC_DATE = pd.to_datetime(INC_DATE)

  return INC_DATE

In [ ]:
df['INC_DATE'] = df.apply(fix_INC_DATE, axis=1)

In [ ]:
df['INC_DATE'].unique()[:10]

<DatetimeArray>
['2025-02-18 23:11:00', '2025-02-18 13:50:00', '2025-02-18 11:13:00',
 '2025-02-18 12:44:00', '2025-02-18 12:45:00', '2025-02-18 17:15:00',
 '2025-02-18 11:30:00', '2025-02-18 15:05:00', '2025-02-18 15:04:00',
 '2025-02-18 01:28:00']
Length: 10, dtype: datetime64[ns]

## Cleaning ```INC_TIME```

In [ ]:
df['INC_TIME'].unique()

array(['23:11', '13:50', '11:13', ..., '11:44', '11:27', '13:16'],
      dtype=object)

Since we already cleaned the date, we can just extract the time component for ```INC_DATE```:

In [ ]:
def fix_INC_TIME(row):
  if pd.isna(row['INC_DATE']):
    return pd.NA

  # Extract the time from INC_DATE and return it
  return row['INC_DATE'].time()

In [ ]:
df['INC_TIME'] = df.apply(fix_INC_TIME, axis=1)

In [ ]:
df['INC_TIME'].unique()

array([datetime.time(23, 11), datetime.time(13, 50),
       datetime.time(11, 13), ..., datetime.time(11, 44),
       datetime.time(11, 27), datetime.time(13, 16)], dtype=object)

## Cleaning ```INC_BEAT```

In [ ]:
df['INC_BEAT'].unique()

array(['Maricopa', '721 Beat', '731 Beat', '213 Beat', '431 Beat',
       '621 Beat', '822 Beat', '831 Beat', '812 Beat', '413 Beat',
       '824 Beat', '423 Beat', '921 Beat', '924 Beat', '232 Beat',
       '223 Beat', '736 Beat', '515 Beat', '711 Beat', '734 Beat',
       '513 Beat', '224 Beat', '813 Beat', '825 Beat', '414 Beat',
       '421 Beat', '412 Beat', '923 Beat', '913 Beat', '911 Beat',
       '723 Beat', '511 Beat', '814', '832 Beat', '821 Beat', '622 Beat',
       '922 Beat', '713 Beat', '726 Beat', '814 Beat', '725 Beat',
       '411 Beat', '932 Beat', '222 Beat', '724 Beat', '823 Beat',
       '625 Beat', '611 Beat', '912 Beat', '424 Beat', '422 Beat',
       '833 Beat', '613 Beat', '634 Beat', '735 Beat', '915 Beat',
       '221 Beat', '834 Beat', '732 Beat', '914 Beat', '722 Beat',
       '231 Beat', '514 Beat', '815 Beat', '233 Beat', '934 Beat',
       '624 Beat', '822', '733 Beat', '933 Beat', '925 Beat', '623 Beat',
       '612 Beat', '614 Beat', '931 Beat', '811 

For consistency, we'll want to remove the 'Beat' substring from all of the values. This is because some include it and other don't, which is why it's best to just remove it.

In [ ]:
def fix_INC_BEAT(row):
  INC_BEAT = row['INC_BEAT']

  # Remove `Beat` substring
  INC_BEAT = INC_BEAT.replace('Beat', '')

  # Also remove `(Airport)` substring
  INC_BEAT = INC_BEAT.replace('(Airport)', '')

  # Strip string
  INC_BEAT = INC_BEAT.strip()

  return INC_BEAT

In [ ]:
df['INC_BEAT'] = df.apply(fix_INC_BEAT, axis=1)

In [ ]:
df['INC_BEAT'].unique()

array(['Maricopa', '721', '731', '213', '431', '621', '822', '831', '812',
       '413', '824', '423', '921', '924', '232', '223', '736', '515',
       '711', '734', '513', '224', '813', '825', '414', '421', '412',
       '923', '913', '911', '723', '511', '814', '832', '821', '622',
       '922', '713', '726', '725', '411', '932', '222', '724', '823',
       '625', '611', '912', '424', '422', '833', '613', '634', '735',
       '915', '221', '834', '732', '914', '722', '231', '514', '815',
       '233', '934', '624', '733', '933', '925', '623', '612', '614',
       '931', '811', '615', '712', '234', '211', '425', '516', '434',
       '631', '512', '214', '432', '714', '632', '435', '633', '715',
       '591', '433', '212', 'Not Available', 'Pinal'], dtype=object)

## Cleaning ```CIT_AGE```

In [ ]:
df['CIT_AGE'].unique()

array(['30', '29', '45', '53', '19', '23', '38', '43', '33', '34', '21',
       '42', '35', '28', '32', '54', '20', '49', '0', '36', '26', '47',
       '41', '44', '31', '65', '37', '50', '48', '55', '46', '15', '71',
       '60', '27', 'Not Available', '39', '25', '40', '63', '5', '3', '1',
       '16', '58', '59', '13', '24', '56', '51', '22', '77', '66', '18',
       '17', '67', '57', '69', '14', '52', '62', '7', '75', '76', '61',
       '72', '64', '68', '81', '70', '78', '12', '9', '79', '74', '82',
       '83'], dtype=object)

We can convert this column to integers, replacing 'Not Available' with NA. Also, we have a few suspicious values that we'll need to take care of. After all, the oldest person to ever live was 122, and so it's unlikely that the 124 and 125 entries are legitimate.

In [ ]:
def fix_CIT_AGE(row):
  CIT_AGE = row['CIT_AGE']

  # For 'Not Available' return NA
  if CIT_AGE == 'Not Available':
    return pd.NA

  # Convert to int
  CIT_AGE = int(CIT_AGE)

  # If age is > 100, return NA
  if CIT_AGE > 100:
    return pd.NA

  return CIT_AGE

In [ ]:
df['CIT_AGE'] = df.apply(fix_CIT_AGE, axis=1)

In [ ]:
df['CIT_AGE'].unique()

array([30, 29, 45, 53, 19, 23, 38, 43, 33, 34, 21, 42, 35, 28, 32, 54, 20,
       49, 0, 36, 26, 47, 41, 44, 31, 65, 37, 50, 48, 55, 46, 15, 71, 60,
       27, <NA>, 39, 25, 40, 63, 5, 3, 1, 16, 58, 59, 13, 24, 56, 51, 22,
       77, 66, 18, 17, 67, 57, 69, 14, 52, 62, 7, 75, 76, 61, 72, 64, 68,
       81, 70, 78, 12, 9, 79, 74, 82, 83], dtype=object)

## Cleaning ```SUBJ_AGE_GROUP```

In [ ]:
df['SUBJ_AGE_GROUP'].unique()

array(['30s', '20s', '40s', '50s', '<20', '60s', '70s', 'Not Available',
       '80s'], dtype=object)

Since we cleaned the ages already, we can just use those to determine the age group:

In [ ]:
def fix_SUBJ_AGE_GROUP(row):
  CIT_AGE = row['CIT_AGE']

  # If age is NA, return NA
  if pd.isna(CIT_AGE):
    return pd.NA
  elif CIT_AGE < 20:
    return '<20'
  elif CIT_AGE < 30:
    return '20s'
  elif CIT_AGE < 40:
    return '30s'
  elif CIT_AGE < 50:
    return '40s'
  elif CIT_AGE < 60:
    return '50s'
  elif CIT_AGE < 70:
    return '60s'
  elif CIT_AGE < 80:
    return '70s'
  elif CIT_AGE < 90:
    return '80s'

  return CIT_AGE

In [ ]:
df['SUBJ_AGE_GROUP'] = df.apply(fix_SUBJ_AGE_GROUP, axis=1)

In [ ]:
df['SUBJ_AGE_GROUP'].unique()

array(['30s', '20s', '40s', '50s', '<20', '60s', '70s', <NA>, '80s'],
      dtype=object)

## Cleaning ```CIT_RACE```

In [ ]:
df['CIT_RACE'].unique()

array(['Black', 'White', 'Black / African American', 'Not Available',
       'American Indian / Alaskan Native', 'Unknown', 'Asian',
       'Native Hawaiian / Other Pacific Islander', 'white', 'Hispanic',
       'Asian / Pacific Islander', 'whi'], dtype=object)

We just need to remove redundant values:

In [ ]:
def fix_CIT_RACE(row):
  CIT_RACE = row['CIT_RACE']

  if 'whi' in CIT_RACE.lower():
    return 'White'
  elif 'black' in CIT_RACE.lower():
    return 'Black'
  elif 'asian' in CIT_RACE.lower():
    return 'Asian'

  return CIT_RACE

In [ ]:
df['CIT_RACE'] = df.apply(fix_CIT_RACE, axis=1)

In [ ]:
df['CIT_RACE'].unique()

array(['Black', 'White', 'Not Available',
       'American Indian / Alaskan Native', 'Unknown', 'Asian',
       'Native Hawaiian / Other Pacific Islander', 'Hispanic'],
      dtype=object)

## Fix ```CIT_ETHNICITY```

In [ ]:
df['CIT_ETHNICITY'].unique()

array(['Non-Hispanic', 'Hispanic / Latino', 'Not Hispanic / Latino',
       'Not Available', 'Unknown', 'Hispanic', 'h'], dtype=object)

Like before, let's remove redundant values:

In [ ]:
def fix_CIT_ETHNICITY(row):
  CIT_ETHNICITY = row['CIT_ETHNICITY']

  if 'non-hispanic' in CIT_ETHNICITY.lower() or 'not hispanic' in CIT_ETHNICITY.lower():
    return 'Non-Hispanic'
  elif 'hispanic' in CIT_ETHNICITY.lower() or 'h' in CIT_ETHNICITY.lower():
    return 'Hispanic'

  return CIT_ETHNICITY

In [ ]:
df['CIT_ETHNICITY'] = df.apply(fix_CIT_ETHNICITY, axis=1)

In [ ]:
df['CIT_ETHNICITY'].unique()

array(['Non-Hispanic', 'Hispanic', 'Not Available', 'Unknown'],
      dtype=object)

## Adjusting Types

# Normalization

In [ ]:
"""
import pandas as pd
!pip install psycopg2 --quiet
import psycopg2

DB_PARAMS = {
    "dbname": "PhoenixDB",
    "user": "postgres",
    "password": "pandabear",
    "host": "localhost",
    "port": "5432"
}


# Once csv is download, replace this line
csv_file = r"C:\DAEN_328\Assignment_3\cleaned_data.csv"

try:
    df = pd.read_csv(csv_file, dtype=str)
    print("✅ CSV file loaded successfully.")
except Exception as e:
    print(f"❌ Error loading CSV file: {e}")
    exit()


column_mapping = {
    "_id": ["_id"],
    "INC_IR_NO": ["INC_IR_NO", "INC_BEAT", "HUNDRED_BLOCK", "INC_CITY", "INC_STATE", "INC_ZIPCODE", "INC_PRECINCT", "HIGHEST_SHOW_FORCE", "SHOW_FORCE_COUNT", "INC_TIME", "INC_YEAR", "INC_DATE", "INC_DAY_WEEK"],
    "CIT_NUMBER":["CIT_NUMBER", "SIMPLE_SUBJ_RE_GRP", "CITIZEN_CHARGE", "CIT_GENDER", "CIT_AGE", "SUBJ_AGE_GROUP", "CIT_RACE", "CIT_ETHNICITY"]

}

print(list(column_mapping.keys()))
print(list(column_mapping.values()))


"""

<>:16: SyntaxWarning: invalid escape sequence '\D'
<>:16: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_11340/1037937547.py:16: SyntaxWarning: invalid escape sequence '\D'
  csv_file = r"C:\DAEN_328\Assignment_3\cleaned_data.csv"


'\nimport pandas as pd\n!pip install psycopg2 --quiet\nimport psycopg2\n\nDB_PARAMS = {\n    "dbname": "PhoenixDB",\n    "user": "postgres",\n    "password": "pandabear",\n    "host": "localhost",\n    "port": "5432"\n}\n\n\n# Once csv is download, replace this line\ncsv_file = r"C:\\DAEN_328\\Assignment_3\\cleaned_data.csv"\n\ntry:\n    df = pd.read_csv(csv_file, dtype=str)\n    print("✅ CSV file loaded successfully.")\nexcept Exception as e:\n    print(f"❌ Error loading CSV file: {e}")\n    exit()\n\n\ncolumn_mapping = {\n    "_id": ["_id"],\n    "INC_IR_NO": ["INC_IR_NO", "INC_BEAT", "HUNDRED_BLOCK", "INC_CITY", "INC_STATE", "INC_ZIPCODE", "INC_PRECINCT", "HIGHEST_SHOW_FORCE", "SHOW_FORCE_COUNT", "INC_TIME", "INC_YEAR", "INC_DATE", "INC_DAY_WEEK"],\n    "CIT_NUMBER":["CIT_NUMBER", "SIMPLE_SUBJ_RE_GRP", "CITIZEN_CHARGE", "CIT_GENDER", "CIT_AGE", "SUBJ_AGE_GROUP", "CIT_RACE", "CIT_ETHNICITY"]\n\n}\n\nprint(list(column_mapping.keys()))\nprint(list(column_mapping.values()))\n\n\n'